In [1]:
from langchain_core.prompts import PromptTemplate
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

In [ ]:
import os
os.environ["COHERE_API_KEY"] = "YOUR_COHERE_API_KEY"

from langchain_cohere import ChatCohere, CohereEmbeddings
llm = ChatCohere()
embeddings = CohereEmbeddings(
  model="embed-english-v3.0"
)

In [3]:
from rich import print

In [4]:
store = {}
# Function to retrieve or create chat history for a given session ID
def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
  if session_id not in store:
    store[session_id] = InMemoryChatMessageHistory()
  messages = store[session_id].messages
  store[session_id] = InMemoryChatMessageHistory(messages=messages)
  return store[session_id]

In [5]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = PyPDFLoader("../langgraph_basics/resume.pdf")   # Change to your PDF path
documents = loader.load()
# Split into chunks for better retrieval
text_splitter = RecursiveCharacterTextSplitter(
  chunk_size=800,
  chunk_overlap=100,
  separators=["\n\n", "\n", ".", " ", ""]
)
texts = text_splitter.split_documents(documents)


In [6]:
from langchain_community.vectorstores import Chroma

# Create Chroma index
db = Chroma.from_documents(texts, embeddings)

In [7]:
retriever=db.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [8]:
chat_prompt = ChatPromptTemplate.from_messages([
  # --- System Message ---
  ("system", """
You are a resume analyst.
Use the retrieved context to answer queries.
If answer for the query is not rooted in the context provided,
answer based on the query appropriately.
Answer any user questions based solely on the context below:
<context>
{context}
</context>
"""),
  ("placeholder", "{chat_history}"),
  ("human", "{input}")
])


In [9]:
from langchain.chains.combine_documents import create_stuff_documents_chain

from langchain.chains import create_retrieval_chain
docs_chain = create_stuff_documents_chain(llm, chat_prompt)
retriever_chain = create_retrieval_chain(
  retriever,
  docs_chain
)


In [10]:
chain_with_memory = RunnableWithMessageHistory(
  retriever_chain,
  get_session_history,
  input_messages_key="input", #input_messages_key → Matches {input} in the ChatPromptTemplate.
  history_messages_key="chat_history",
  output_messages_key='answer' #history_messages_key → Matches {chat_history} in the ChatPromptTemplate.
)

In [11]:
session_id = "session_1"
queries = [
  "Who is the resume about?",
  "What are the key skills mentioned in the resume?",
]

In [12]:
for i, q in enumerate(queries, 1):
  response = chain_with_memory.invoke(
    {"input": q},
    config={"configurable": {"session_id": session_id}}
  )
  print(f"\nQuery {i}: {q}")
  print("Answer:", response["answer"])

Query 1: Who is the resume about?

Answer: The resume is about **Divyanshu Yadav**, a **Software Engineer** based in **Hyderabad, India**. The resume 
highlights his education, experience, and technical skills, including projects he has worked on, such as building a
keyword analytics platform, an AI-powered support chatbot, an AI-powered hiring integrity platform, and an Agentic 
AI-powered multilingual invoice auditing system. His contact information, including email and phone number, is also
provided.

Query 2: What are the key skills mentioned in the resume?

Answer: The key skills mentioned in the resume include:

1. **Programming Languages**:  
   - Java  
   - TypeScript  
   - JavaScript  
   - Python  

2. **Libraries/Frameworks**:  
   - React.js  
   - Next.js  
   - Express  
   - Flutter  
   - OpenCV  
   - FastAPI  
   - Streamlit  

3. **Tools/Platforms**:  
   - Google Search Console APIs  
   - WebRTC  

4. **Databases**:  
   Not explicitly mentioned in the provided context.  

5. **Other Technical Skills**:  
   - Semantic retrieval and personalized answer generation  
   - Computer vision  
   - RAG (Retrieval-Augmented Generation) pipeline development  
   - AI-powered chatbot development  
   - SEO optimization  
   - Migration from React to Next.js (SSG + pre-rendering)  

These skills are demonstrated through various projects and achievements listed in the resume.